# 02 · Wildfires — burn-scar segmentation (HLS Burn Scars)

**CRISP module:** Wildfires · **Task:** per-pixel burn-scar segmentation · **Model:** U-Net / ResNet-34

Input bands are the six Sentinel-2-equivalent HLS bands (Blue, Green, Red, NIR-narrow, SWIR-1, SWIR-2), so the
model can be applied directly to Sentinel-2 imagery of the CRISP wildfire sites. The official `validation` folder is
held out as the **test** set; validation for early stopping is carved from `training`, grouped by Sentinel-2 tile so that
neighbouring chips never sit on both sides. Compared against an NBR threshold — CRISP's current burn rule.
Runtime on a T4: about 20 min.

In [ ]:
# ── Setup: run once per Colab session (Runtime ▸ Change runtime type ▸ T4 GPU) ──
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/inayatarshad/trainedmodels_gstp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content")
    DRIVE = Path("/content/drive/MyDrive/gstp_runs")       # checkpoints + results persist here
    REPO = WORK / "trainedmodels_gstp"
    if not REPO.exists():
        if subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)]).returncode != 0:
            # Private repo? Upload the repo zip to MyDrive/trainedmodels_gstp.zip and re-run.
            shutil.unpack_archive("/content/drive/MyDrive/trainedmodels_gstp.zip", WORK)
            REPO = next(WORK.glob("trainedmodels_gstp*"))
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:  # local / Kaggle
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO)
    WORK = DRIVE = REPO / "runs"

sys.path.insert(0, str(REPO / "src"))
DRIVE.mkdir(parents=True, exist_ok=True)

import torch
from gstp.utils import seed_everything, get_device
seed_everything(42)
print("Repo:", REPO, "| device:", get_device(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
MODULE = "wildfires"
from gstp.data import burn_scars as B
from gstp.data.segmentation import SegDataset, split_samples, read_sample_image
from gstp.data.raster import compute_band_stats
from gstp.models import build_unet, count_parameters, export_onnx
from gstp import train_seg, baselines, viz
from gstp.utils import save_json
import numpy as np, pandas as pd, torch, tarfile

DATA = WORK / "data" / "hls_burn_scars"
RESULTS = REPO / "results" / MODULE
CKPT_DIR = DRIVE / MODULE
for p in (DATA, RESULTS / "figures", CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)

## 1. Download from Hugging Face

In [ ]:
from huggingface_hub import snapshot_download
if not any(DATA.rglob("*merged*.tif")):
    snapshot_download("ibm-nasa-geospatial/hls_burn_scars", repo_type="dataset", local_dir=str(DATA))
    for t in DATA.rglob("*.tar.gz"):
        with tarfile.open(t) as tf:
            tf.extractall(DATA)
print("chips:", len(list(DATA.rglob("*merged*.tif"))))

In [ ]:
train_all = B.build_samples(DATA, "training")
test = B.build_samples(DATA, "validation")
assert train_all and test, "Burn-scar chips not found — check the extracted folder layout"
train, val = split_samples(train_all, 0.15, seed=42, group_key=lambda s: s.meta["tile"])
print(f"train {len(train)} | val {len(val)} | test {len(test)}")
stats = compute_band_stats(train, read_sample_image, nodata=B.NODATA)
units_scale = 10000.0 if np.median(stats["mean"]) > 10 else 1.0   # reflectance units of this release
print("band means:", np.round(stats["mean"], 4), "| reflectance scale:", units_scale)
save_json({"stats": stats, "reflectance_scale": units_scale}, RESULTS / "band_stats.json")

## 2. Train

In [ ]:
cfg = train_seg.SegTrainConfig(name="burn_scar", in_channels=6, bands=B.BANDS, epochs=60, batch_size=16,
                               crop_size=256, lr=3e-4, patience=12,
                               extra={"norm": "zscore", "stats": stats, "reflectance_scale": units_scale,
                                      "resolution_m": 30, "dataset": "HLS Burn Scars"})
tr = SegDataset(train, "zscore", stats, nodata=B.NODATA, train=True, crop_size=256)
va = SegDataset(val, "zscore", stats, nodata=B.NODATA)
model = build_unet(6)
print(f"{count_parameters(model)/1e6:.1f} M parameters")
summary = train_seg.train(model, tr, va, cfg, CKPT_DIR)
viz.training_curves(summary["history"], title="burn_scar", path=RESULTS / "figures" / "curves_burn_scar.png");

## 3. Test once, and compare with the NBR-threshold rule

In [ ]:
thr = summary["chosen_threshold"]
report = train_seg.test_report(model, {"test": SegDataset(test, "zscore", stats, nodata=B.NODATA)}, thr,
                               RESULTS / "test_burn_scar.json")
NIR, SW2 = B.BANDS.index("B8A"), B.BANDS.index("B12")
bl = baselines.tune_and_test(val, {"test": test}, lambda x: baselines.normalized_difference(x[NIR], x[SW2]), "<",
                             nodata=B.NODATA)
save_json({"NBR threshold": bl}, RESULTS / "baselines.json")
rows = [{"model": "U-Net (6 bands)", **{k: report["test"]["at_chosen_threshold"][k] for k in ("iou", "f1", "precision", "recall", "miou")}},
        {"model": f"NBR < {bl['threshold']:.3f}", **{k: bl["test"][k] for k in ("iou", "f1", "precision", "recall", "miou")}}]
table = pd.DataFrame(rows).rename(columns={"iou": "IoU (burn)"})
table.to_csv(RESULTS / "results_table.csv", index=False)
table.style.format(precision=3).hide(axis="index")

In [ ]:
val_meter, _ = train_seg.evaluate(model, va)
viz.threshold_curve(val_meter, thr, "burn_scar: validation", RESULTS / "figures" / "threshold_curve.png")
ds, raw = SegDataset(test, "zscore", stats, nodata=B.NODATA), SegDataset(test, "none", nodata=B.NODATA)
order = np.argsort([-np.mean(ds.load(i)[1] == 1) for i in range(len(ds))])[:20:5]
ex = []
for i in order:
    x, y = ds.load(i)
    with torch.no_grad():
        p = torch.sigmoid(model.eval()(torch.from_numpy(x)[None].to(get_device()))).squeeze().cpu().numpy()
    img, _ = raw.load(i)
    ex.append({"rgb": viz.rgb(img, (5, 3, 2)), "mask": y, "prob": p, "thr": thr, "name": ds.samples[i].id[-30:]})
viz.prediction_grid(ex, "Burn scars — test (false colour SWIR2/NIR/Red)", RESULTS / "figures" / "examples_test.png",
                    color=(1.0, 0.3, 0.0));

In [ ]:
export_onnx(model, 6, 512, str(CKPT_DIR / "burn_scar.onnx"))
save_json({"summary": {k: v for k, v in summary.items() if k != "history"}, "test": report, "baseline": bl},
          RESULTS / "summary.json")

In [ ]:
# ── Package results: metrics + figures go to Drive as a zip you can download ──
import shutil
archive = shutil.make_archive(str(DRIVE / f"results_{MODULE}"), "zip", RESULTS)
print("Results zip :", archive)
print("Checkpoints :", sorted(str(p) for p in CKPT_DIR.glob("*.pt")))